# CREAM Noisy Graph Analysis

Single CREAM model trained on one noisy/perturbed expert graph (expert_0).
Baseline comparison for mCREAM Graph Ensemble experiments.

## Three sections:
1. **CREAM Baseline** — GT graph, best config
2. **CREAM on Noisy Graphs** — addition/deletion/reversal × low/medium/high (expert_0)
3. **CREAM on Edge Count Graphs** — u2c ±5 from GT=17 (5 seeds per count)

In [ ]:
import pandas as pd, numpy as np, re, torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ACTION_COLOR = {'deletion': '#e74c3c', 'addition': '#2ecc71', 'reversal': '#3498db'}
LEVEL_COLOR  = {'low': '#3498db', 'medium': '#e67e22', 'high': '#e74c3c'}
LEVEL_NUM    = {'low': 0.25, 'medium': 0.50, 'high': 0.75}

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')
GRAPHS_ROOT      = Path('/home/dani00003/mCREAM/data/FashionMNIST')
DAG_CFMNIST      = GRAPHS_ROOT / 'Complete_Concept_FMNIST_DAG.csv'

DATASET  = 'Complete_Concept_FMNIST'
MODEL    = 'Standard_FashionMNIST'
ACTIONS  = ['deletion', 'addition', 'reversal']
LEVELS   = ['low', 'medium', 'high']
GT_EDGES = 17

# ── Shared helpers ────────────────────────────────────────────────────────────
def load_csv_results(version_dir):
    rows = []
    for csv_f in sorted(Path(version_dir).glob('*.csv')):
        if any(x in csv_f.name for x in ['perc_', '_set_', 'intervention', 'exogenous']):
            continue
        try: rows.append(pd.read_csv(csv_f))
        except: pass
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def load_intervention_csv(exp_dir):
    for csv_f in Path(exp_dir).rglob('intervention_results.csv'):
        try: return pd.read_csv(csv_f)
        except: pass
    return pd.DataFrame()

def iter_seeds(exp_dir):
    """Yield (seed, version_dir) for every seed_*/lightning_logs/version_* under exp_dir."""
    for seed_dir in sorted(Path(exp_dir).glob('seed_*/lightning_logs/version_*')):
        seed = int(seed_dir.parent.parent.name.split('_')[1])
        yield seed, seed_dir

print(f'Root exists: {EXPERIMENTS_ROOT.exists()}')

---
# Section 1 — CREAM Baseline (GT graph)

Single CREAM on ground-truth DAG. Upper bound for all comparisons below.  
Config: `all_configs/best_hparams/CREAM/CREAM_best_cfmnist_soft_config.yaml`  
Job: `bash server_scripts/cream_experiment/submit_cream_baseline.sh`

In [ ]:
def load_baseline(root):
    rows = []
    for exp_name in ['CREAM_best_cfmnist', 'CREAM_best_cfmnist_soft']:
        exp_dir = root / DATASET / 'train_cbm' / MODEL / exp_name
        if not exp_dir.exists(): continue
        for seed, vdir in iter_seeds(exp_dir):
            df = load_csv_results(vdir)
            if df.empty: continue
            df['seed'] = seed
            rows.append(df)
    if not rows: print('No baseline results. Submit: submit_cream_baseline.sh'); return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    print(f'Baseline: {len(df)} rows, {df.seed.nunique()} seeds')
    return df

baseline_df = load_baseline(EXPERIMENTS_ROOT)

# Store reference values for dashed lines in later sections
baseline_task_acc    = baseline_df['test_task_accuracy'].mean()    if len(baseline_df) and 'test_task_accuracy'    in baseline_df else None
baseline_concept_acc = baseline_df['test_concept_accuracy'].mean() if len(baseline_df) and 'test_concept_accuracy' in baseline_df else None
baseline_cci         = baseline_df['CCI'].mean()                   if len(baseline_df) and 'CCI'                   in baseline_df else None
print(f'Baseline refs  task={baseline_task_acc}  concept={baseline_concept_acc}  CCI={baseline_cci}')

In [ ]:
if len(baseline_df) == 0:
    print('No baseline data yet.')
else:
    KEY = ['test_task_accuracy', 'test_concept_accuracy', 'CCI', 'PFI_concept_importance']
    av  = [c for c in KEY if c in baseline_df.columns]
    print('=== CREAM Baseline ===')
    display(baseline_df[av].describe().round(4))

    fig, axes = plt.subplots(1, len(av), figsize=(4*len(av), 4))
    if len(av) == 1: axes = [axes]
    fig.suptitle('CREAM Baseline — GT graph (cfmnist)', fontsize=12, fontweight='bold')
    for ax, metric in zip(axes, av):
        vals = baseline_df[metric].dropna()
        ax.bar(['CREAM\nbaseline'], [vals.mean()], yerr=[vals.std()],
               color='#2c3e50', alpha=0.75, capsize=6, error_kw={'lw': 2})
        ax.set_title(metric.replace('test_','').replace('_',' '), fontsize=9)
        ax.set_ylim(0, min(vals.mean()*1.2 + 0.01, 1.1))
    plt.tight_layout()
    plt.savefig('cream_baseline_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Section 2 — CREAM on Noisy Graphs (expert_0)

One CREAM per (action, level) trained on `noisy_dag_{action}_{level}.csv` (expert_0).  
9 experiments: addition/deletion/reversal × low/medium/high, 5 training seeds each.

## Two intervention modes (shown separately)

**Individual concept interventions** (`group_interventions=False`):
- Replace one concept at a time, randomly chosen
- x-axis: 0 → 11 (one per concept)
- Answers: *how much does each extra concept fix help?*

**Group interventions** (`group_interventions=True`):
- Replace all concepts in one mutex group at once (cfmnist has 3 groups)
- x-axis: 0 → 3 (one per mutex group: `[0,5]`, `[1,2,3,4,6,7]`, `[8,9,10]`)
- One step = revealing all concepts in a semantic category together
- Answers: *how much does fixing a whole semantic group help?*
- Steeper curve = that semantic group is more important for the task

**Prerequisites:**
```bash
python generate_ensemble_expert_graphs.py --dataset cfmnist
python generate_single_noisy_dags.py --dataset cfmnist --action addition --level low   # ×9
bash server_scripts/cream_experiment/submit_cream_noisy.sh
```

In [ ]:
def load_noisy(root):
    rows, irows = [], []
    exp_root = root / DATASET / 'train_cbm' / MODEL
    if not exp_root.exists(): print(f'Not found: {exp_root}'); return pd.DataFrame(), pd.DataFrame()
    for exp_dir in sorted(exp_root.iterdir()):
        m = re.match(r'cream_noisy_(addition|deletion|reversal)_(low|medium|high)', exp_dir.name)
        if not m: continue
        action, level = m.group(1), m.group(2)
        for seed, vdir in iter_seeds(exp_dir):
            df = load_csv_results(vdir)
            if df.empty: continue
            df['action'] = action; df['noise_level'] = level; df['seed'] = seed
            rows.append(df)
        # interventions
        for seed_dir in sorted(exp_dir.glob('seed_*')):
            seed = int(seed_dir.name.split('_')[1])
            iv = load_intervention_csv(seed_dir)
            if iv.empty: continue
            iv['action'] = action; iv['noise_level'] = level; iv['seed'] = seed
            irows.append(iv)
    df  = pd.concat(rows,  ignore_index=True) if rows  else pd.DataFrame()
    ivdf= pd.concat(irows, ignore_index=True) if irows else pd.DataFrame()
    if len(df): print(f'Noisy: {len(df)} rows | actions={sorted(df.action.unique())} | levels={sorted(df.noise_level.unique())}')
    else: print('No noisy results yet.')
    return df, ivdf

noisy_df, noisy_interv = load_noisy(EXPERIMENTS_ROOT)

In [ ]:
# Show expert_0 graph for each action × level
graphs_base = GRAPHS_ROOT / 'expert_graphs' / 'ensemble'
gt_adj = (pd.read_csv(DAG_CFMNIST, index_col=0).values != 0).astype(int)[:11, :11]

for action in ACTIONS:
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    fig.suptitle(f'CREAM noisy — {action}: expert_0 graph per level  '
                 f'(blue=kept, red=deleted, green=added)', fontsize=10, fontweight='bold')
    for ax, level in zip(axes, LEVELS):
        pt = graphs_base / f'{action}_{level}' / 'u2c' / 'expert_0.pt'
        if not pt.exists(): ax.text(0.5,0.5,'N/A',ha='center',va='center',transform=ax.transAxes); ax.axis('off'); continue
        p   = torch.load(pt, weights_only=True).float().numpy()
        rgb = np.zeros((*gt_adj.shape, 3))
        for r in range(11):
            for c in range(11):
                if   gt_adj[r,c]==1 and p[r,c]>0.5: rgb[r,c]=[0.2,0.4,0.8]
                elif gt_adj[r,c]==0 and p[r,c]>0.5: rgb[r,c]=[0.2,0.8,0.2]
                elif gt_adj[r,c]==1 and p[r,c]<=0.5: rgb[r,c]=[0.9,0.2,0.2]
                else: rgb[r,c]=[0.93,0.93,0.93]
        ax.imshow(rgb, aspect='auto', interpolation='nearest')
        ax.set_title(level, fontsize=9, color=LEVEL_COLOR[level], fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.savefig(f'cream_noisy_dag_{action}.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
if len(noisy_df) == 0:
    print('No noisy results yet — see prerequisites above.')
else:
    KEY = ['test_task_accuracy', 'test_concept_accuracy', 'CCI', 'PFI_concept_importance']
    av  = [c for c in KEY if c in noisy_df.columns]

    # Summary table
    means = noisy_df.groupby(['action','noise_level'])[av].mean()
    stds  = noisy_df.groupby(['action','noise_level'])[av].std()
    sm = pd.DataFrame({c: means[c].map('{:.4f}'.format)+' ± '+stds[c].map('{:.4f}'.format) for c in av})
    print('=== Section 2: CREAM on Noisy Graphs ===')
    display(sm)

    # Line plot per metric
    for metric, ylabel in [('test_task_accuracy','Task Accuracy'),
                            ('test_concept_accuracy','Concept Accuracy'),
                            ('CCI','CCI')]:
        if metric not in noisy_df.columns: continue
        fig, ax = plt.subplots(figsize=(7, 4.5))
        fig.suptitle(f'CREAM on Noisy Graphs — {ylabel} vs noise level', fontsize=11, fontweight='bold')
        d = noisy_df.copy(); d['noise_prob'] = d['noise_level'].map(LEVEL_NUM)
        agg = d.groupby(['action','noise_prob'])[metric].agg(['mean','std']).reset_index()
        for action in ACTIONS:
            sub = agg[agg['action']==action]
            if sub.empty: continue
            col = ACTION_COLOR[action]
            ax.plot(sub['noise_prob'], sub['mean'], color=col, marker='o', markersize=8, lw=2, label=action)
            ax.fill_between(sub['noise_prob'], sub['mean']-sub['std'], sub['mean']+sub['std'], alpha=0.12, color=col)
        ref = {'test_task_accuracy': baseline_task_acc, 'test_concept_accuracy': baseline_concept_acc, 'CCI': baseline_cci}.get(metric)
        if ref: ax.axhline(ref, color='black', lw=2, ls='--', label=f'GT CREAM: {ref:.4f}')
        ax.set_xlabel('Noise probability'); ax.set_ylabel(ylabel)
        ax.set_xticks([0.25,0.50,0.75]); ax.set_xticklabels(['low','medium','high'])
        ax.legend(title='Noise type', fontsize=9)
        plt.tight_layout(); plt.savefig(f'cream_noisy_{metric}.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
if len(noisy_interv) == 0:
    print('No noisy intervention data yet.')
else:
    has_group_col = 'group_interventions' in noisy_interv.columns

    for action in ACTIONS:
        sub_all = noisy_interv[noisy_interv['action']==action]
        if sub_all.empty: continue

        sub_indiv = sub_all[sub_all['group_interventions']==False] if has_group_col else sub_all
        sub_group = sub_all[sub_all['group_interventions']==True]  if has_group_col else pd.DataFrame()

        fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey='row')
        fig.suptitle(
            f'CREAM noisy — {action}: Intervention curves\n'
            f'TOP: individual concept (0→11)   BOTTOM: group/mutex (0→3)\n'
            f'Band = ± std across 5 training seeds',
            fontsize=11, fontweight='bold'
        )

        for col_idx, level in enumerate(LEVELS):
            col = LEVEL_COLOR[level]

            # ── TOP ROW: individual ───────────────────────────────────────────
            ax_top = axes[0][col_idx]
            lv = sub_indiv[sub_indiv['noise_level']==level]
            if not lv.empty:
                agg = lv.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                ax_top.plot(agg['num_interventions'], agg['mean'], color=col, lw=2.5, marker='o', markersize=5)
                ax_top.fill_between(agg['num_interventions'], agg['mean']-agg['std'], agg['mean']+agg['std'],
                                    alpha=0.15, color=col)
            ax_top.set_title(f'{level} — individual', fontsize=10, color=LEVEL_COLOR[level], fontweight='bold')
            ax_top.set_xlabel('Concepts replaced (0→11)', fontsize=8)
            ax_top.set_ylabel('Task Accuracy' if col_idx==0 else '', fontsize=9)
            ax_top.tick_params(labelsize=8)

            # ── BOTTOM ROW: group ─────────────────────────────────────────────
            ax_bot = axes[1][col_idx]
            if not sub_group.empty:
                lv_g = sub_group[sub_group['noise_level']==level]
                if not lv_g.empty:
                    agg_g = lv_g.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                    ax_bot.plot(agg_g['num_interventions'], agg_g['mean'], color=col, lw=2.5, marker='s',
                                markersize=6, ls='--')
                    ax_bot.fill_between(agg_g['num_interventions'], agg_g['mean']-agg_g['std'],
                                        agg_g['mean']+agg_g['std'], alpha=0.15, color=col)
            ax_bot.set_title(f'{level} — group (mutex)', fontsize=10, color=LEVEL_COLOR[level], fontweight='bold')
            ax_bot.set_xlabel('Mutex groups replaced (0→3)', fontsize=8)
            ax_bot.set_ylabel('Task Accuracy' if col_idx==0 else '', fontsize=9)
            ax_bot.tick_params(labelsize=8)

        plt.tight_layout()
        plt.savefig(f'cream_noisy_interv_{action}.png', dpi=150, bbox_inches='tight')
        plt.show()

---
# Section 3 — CREAM on Edge Count Graphs (±5 from GT=17)

One CREAM per (edge_count, graph_seed) — 11 counts × 5 seeds = 55 models.  
Each graph_seed is a different random edge selection at the same count.  

Intervention plots show both modes side by side — same layout as Section 2.

**Prerequisites:**
```bash
bash server_scripts/cream_experiment/submit_cream_edge_count.sh
```

In [ ]:
def load_edge_count(root):
    rows, irows = [], []
    exp_root = root / DATASET / 'train_cbm' / MODEL
    if not exp_root.exists(): print(f'Not found: {exp_root}'); return pd.DataFrame(), pd.DataFrame()
    for exp_dir in sorted(exp_root.iterdir()):
        m = re.match(r'edge_count_(u2c|c2y)_(\d+)edges_seed(\d+)', exp_dir.name)
        if not m: continue
        gtype, n_edges, gseed = m.group(1), int(m.group(2)), int(m.group(3))
        for seed, vdir in iter_seeds(exp_dir):
            df = load_csv_results(vdir)
            if df.empty: continue
            df['graph_type'] = gtype; df['edge_count'] = n_edges
            df['graph_seed'] = gseed; df['train_seed'] = seed
            rows.append(df)
        for seed_dir in sorted(exp_dir.glob('seed_*')):
            seed = int(seed_dir.name.split('_')[1])
            iv = load_intervention_csv(seed_dir)
            if iv.empty: continue
            iv['graph_type'] = gtype; iv['edge_count'] = n_edges
            iv['graph_seed'] = gseed; iv['train_seed'] = seed
            irows.append(iv)
    df  = pd.concat(rows,  ignore_index=True) if rows  else pd.DataFrame()
    ivdf= pd.concat(irows, ignore_index=True) if irows else pd.DataFrame()
    if len(df): print(f'Edge count: {len(df)} rows | counts={sorted(df.edge_count.unique())}')
    else: print('No edge count results yet.')
    return df, ivdf

ec_df, ec_interv = load_edge_count(EXPERIMENTS_ROOT)

In [ ]:
if len(ec_df) == 0:
    print('No edge count results yet — see prerequisites above.')
else:
    KEY = ['test_task_accuracy', 'test_concept_accuracy', 'CCI']
    av  = [c for c in KEY if c in ec_df.columns]

    for metric, ylabel in [(m, m.replace('test_','').replace('_',' ').title()) for m in av]:
        fig, ax = plt.subplots(figsize=(10, 5))
        fig.suptitle(f'CREAM on Edge Count Graphs — {ylabel}\n'
                     f'(GT=17, boxplot = 5 different random graphs per count)',
                     fontsize=11, fontweight='bold')

        # Average across train seeds first, then boxplot across graph seeds
        agg = ec_df[ec_df['graph_type']=='u2c'].groupby(['edge_count','graph_seed'])[metric].mean().reset_index()
        edge_counts = sorted(agg['edge_count'].unique())
        boxes = [agg[agg['edge_count']==c][metric].values for c in edge_counts]

        cmap = plt.cm.RdYlGn
        bp = ax.boxplot(boxes, positions=edge_counts, widths=0.6,
                        patch_artist=True, medianprops=dict(color='black', lw=2))
        for patch, count in zip(bp['boxes'], edge_counts):
            norm = (count - min(edge_counts)) / max(max(edge_counts)-min(edge_counts), 1)
            patch.set_facecolor(cmap(norm)); patch.set_alpha(0.7)

        ax.axvline(x=GT_EDGES, color='red', ls=':', lw=2, alpha=0.8, label=f'GT count ({GT_EDGES})')
        refs = {'test_task_accuracy': baseline_task_acc, 'test_concept_accuracy': baseline_concept_acc, 'CCI': baseline_cci}
        if refs.get(metric): ax.axhline(refs[metric], color='black', lw=2, ls='--', label=f'GT CREAM: {refs[metric]:.4f}')

        ax.set_xlabel('Number of u2c edges', fontsize=10); ax.set_ylabel(ylabel, fontsize=10)
        ax.set_xticks(edge_counts); ax.legend(fontsize=9)
        plt.tight_layout(); plt.savefig(f'cream_edge_count_{metric}.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
if len(ec_interv) == 0:
    print('No edge count intervention data yet.')
else:
    sub_ec = ec_interv[ec_interv['graph_type']=='u2c']
    has_group_col = 'group_interventions' in sub_ec.columns
    sub_indiv = sub_ec[sub_ec['group_interventions']==False] if has_group_col else sub_ec
    sub_group = sub_ec[sub_ec['group_interventions']==True]  if has_group_col else pd.DataFrame()

    show_counts = [c for c in [12, 15, 17, 19, 22] if c in sub_indiv['edge_count'].unique()]
    if not show_counts: show_counts = sorted(sub_indiv['edge_count'].unique())
    all_counts = sorted(sub_indiv['edge_count'].unique())
    cmap = plt.cm.RdYlGn

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        'CREAM on Edge Count Graphs — Intervention Curves\n'
        f'(GT=17, band = ±std across seeds)\n'
        f'LEFT: individual concept (0→11)   RIGHT: group/mutex (0→3)',
        fontsize=11, fontweight='bold'
    )

    for ax, src, xlabel, title_suffix in [
        (axes[0], sub_indiv, 'Concepts replaced (0→11)',   'Individual interventions'),
        (axes[1], sub_group, 'Mutex groups replaced (0→3)', 'Group interventions'),
    ]:
        if src.empty: ax.set_title(f'{title_suffix} — no data'); ax.axis('off'); continue
        sc = [c for c in show_counts if c in src['edge_count'].unique()]
        for count in sc:
            lv  = src[src['edge_count']==count]
            agg = lv.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
            norm  = all_counts.index(count) / max(len(all_counts)-1, 1)
            color = cmap(norm)
            style = '--' if count < GT_EDGES else ('-' if count==GT_EDGES else '-.')
            lw    = 3 if count==GT_EDGES else 1.8
            ax.plot(agg['num_interventions'], agg['mean'], color=color, lw=lw, ls=style,
                    marker='o', markersize=4,
                    label=f'{count} edges{" (GT)" if count==GT_EDGES else ""}')
            ax.fill_between(agg['num_interventions'], agg['mean']-agg['std'], agg['mean']+agg['std'],
                            alpha=0.08, color=color)
        ax.axhline(1.0, color='gray', ls=':', alpha=0.4, lw=1)
        ax.set_title(title_suffix, fontsize=10)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel('Task Accuracy', fontsize=9)
        ax.legend(fontsize=7, loc='lower right', framealpha=0.95)
        ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig('cream_edge_count_interventions.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Cross-Section Summary

Baseline vs noisy vs edge-perturbed in one table.

---
# Intervention Debug Log Reader

Reads the `intervention_debug_*.txt` files written during testing.  
Each block shows one test batch — what `c` looked like before/after replacement.

## What each field means

| Field | What it is |
|---|---|
| `c BEFORE intervention` | Model's predicted concept vector — what the model thinks concepts are |
| `true_concepts` | Ground-truth concepts scaled to model's activation range (95th pct for 1, 5th pct for 0) |
| `diff (c - true)` | How wrong the model was — large values = model was confused on that concept |
| `intervention_mask` | Which concepts were replaced this step (True = replaced) |
| `Group sums` | Each mutex group should sum to 1.0. If sum=0 after replacement → label says no class active (noisy label) |

## How to read the intervention curve from this

At n=0: `c BEFORE` = model prediction, no replacement → baseline accuracy  
At n=1: one random concept replaced → accuracy changes  
At n=11: all concepts replaced → upper bound accuracy

If `diff` values are small before intervention → model was already close → curve rises slowly  
If `diff` values are large → model was wrong on many concepts → curve rises steeply when corrected

In [ ]:
def parse_debug_log(log_path, max_blocks=5):
    """Parse intervention_debug_*.txt and print annotated blocks."""
    log_path = Path(log_path)
    if not log_path.exists():
        print(f'Log not found: {log_path}')
        print('Enable with: model._debug_interventions = True  before trainer.test()')
        return

    text = log_path.read_text()
    blocks = [b.strip() for b in text.split('='*60) if b.strip()]
    print(f'Found {len(blocks)} intervention blocks in {log_path.name}')
    print(f'Showing first {min(max_blocks, len(blocks))} blocks\n')

    for i, block in enumerate(blocks[:max_blocks]):
        lines = block.strip().split('\n')
        print(f'{'─'*70}')
        print(f'BLOCK {i+1}')

        c_before = true_c = diff = mask = None
        for line in lines:
            if 'num_interventions' in line:
                print(f'  {line.strip()}')
            elif 'BEFORE' in line and 'c' in line.lower():
                # next line has the values
                pass
            elif c_before is None and line.strip().startswith('[') and 'BEFORE' in ''.join(lines[:lines.index(line)]):
                c_before = eval(line.strip())
            elif 'true_concepts' in line.lower():
                pass
            elif true_c is None and line.strip().startswith('[') and 'true' in ''.join(lines[:lines.index(line)]).lower():
                true_c = eval(line.strip())
            elif 'diff' in line.lower() and line.strip().startswith('['):
                diff = line.strip()
            elif 'intervention_mask' in line.lower():
                pass
            elif mask is None and line.strip().startswith('[') and 'mask' in ''.join(lines[:lines.index(line)]).lower():
                mask = eval(line.strip())
            elif 'Group sums' in line or 'group' in line.lower():
                print(f'  {line.strip()}')

        if c_before:
            print(f'\n  c predicted (before):  {[f"{v:.3f}" for v in c_before]}')
        if true_c:
            print(f'  true concepts:         {[f"{v:.3f}" for v in true_c]}')
        if diff:
            print(f'  diff (c - true):       {diff}')
        if mask:
            replaced = [i for i,v in enumerate(mask) if v]
            print(f'  concepts replaced:     {replaced}  ({len(replaced)} concepts)')

        if c_before and true_c:
            avg_err = sum(abs(c_before[j] - true_c[j]) for j in range(len(c_before))) / len(c_before)
            print(f'\n  → avg |c - true| = {avg_err:.4f}  '
                  f'({"model is close to truth" if avg_err < 0.05 else "model is wrong on some concepts"})')
        print()

# ── Adjust paths to match your server log locations ───────────────────────────
CREAM_LOG  = Path('/home/dani00003/mCREAM/experiments/intervention_debug_cream.txt')
MCREAM_LOG = Path('/home/dani00003/mCREAM/experiments/intervention_debug_graph_ensemble.txt')

print('=== CREAM debug log ===')
parse_debug_log(CREAM_LOG, max_blocks=3)

print('\n=== mCREAM Graph Ensemble debug log ===')
parse_debug_log(MCREAM_LOG, max_blocks=3)

In [ ]:
rows = []

if len(baseline_df):
    r = {'section':'Baseline','experiment':'GT graph','action':'—','level':'—'}
    for m in ['test_task_accuracy','test_concept_accuracy','CCI']:
        r[m] = baseline_df[m].mean() if m in baseline_df.columns else float('nan')
    rows.append(r)

if len(noisy_df):
    for (action, level), grp in noisy_df.groupby(['action','noise_level']):
        r = {'section':'Noisy','experiment':f'{action}_{level}','action':action,'level':level}
        for m in ['test_task_accuracy','test_concept_accuracy','CCI']:
            r[m] = grp[m].mean() if m in grp.columns else float('nan')
        rows.append(r)

if len(ec_df):
    for count, grp in ec_df[ec_df['graph_type']=='u2c'].groupby('edge_count'):
        r = {'section':'Edge Count','experiment':f'u2c_{count}edges','action':'u2c','level':str(count)}
        for m in ['test_task_accuracy','test_concept_accuracy','CCI']:
            r[m] = grp[m].mean() if m in grp.columns else float('nan')
        rows.append(r)

if rows:
    sm = pd.DataFrame(rows)
    print('=== Cross-section summary ===')
    display(sm.round(4))
    sm.to_csv('cream_noisy_summary.csv', index=False)
    print('Saved: cream_noisy_summary.csv')
else:
    print('No data in any section yet.')